# Model Definition and Evaluation
## Table of Contents
1. [Model Selection](#model-selection)
2. [Feature Engineering](#feature-engineering)
3. [Hyperparameter Tuning](#hyperparameter-tuning)
4. [Implementation](#implementation)
5. [Evaluation Metrics](#evaluation-metrics)
6. [Comparative Analysis](#comparative-analysis)


In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, mean_squared_error, classification_report
# Import models you're considering


## Model Selection

[Discuss the type(s) of models you consider for this task, and justify the selection.]



## Feature Engineering

[Describe any additional feature engineering you've performed beyond what was done for the baseline model.]


In [ ]:
from pathlib import Path

# Load the dataset
DATA_PATH = Path("..") / "1_DatasetCharacteristics" / "landmarks" / "landmarks_all_10fps.csv"
df = pd.read_csv(DATA_PATH)

# Feature and target variable selection
feature_cols = [c for c in df.columns if c not in ("Good Stroke", "Bad Stroke")]
X = df[feature_cols].values
y = df["Good Stroke"].values  # 1 = GOOD, 0 = BAD

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


## Hyperparameter Tuning

[Discuss any hyperparameter tuning methods you've applied, such as Grid Search or Random Search, and the rationale behind them.]


In [ ]:
# Implement hyperparameter tuning
# Example using GridSearchCV with a DecisionTreeClassifier
# param_grid = {'max_depth': [2, 4, 6, 8]}
# grid_search = GridSearchCV(DecisionTreeClassifier(), param_grid, cv=5)
# grid_search.fit(X_train, y_train)


## Implementation

[Implement the final model(s) you've selected based on the above steps.]


In [ ]:
# Implement the final model(s)
# Example: model = YourChosenModel(best_hyperparameters)
# model.fit(X_train, y_train)


## Evaluation Metrics

[Clearly specify which metrics you'll use to evaluate the model performance, and why you've chosen these metrics.]


In [ ]:
# Evaluate the model using your chosen metrics
# Example for classification
# y_pred = model.predict(X_test)
# print(classification_report(y_test, y_pred))

# Example for regression
# mse = mean_squared_error(y_test, y_pred)

# Your evaluation code here


## Comparative Analysis

Two models, two different framings of the task. The Random Forest ([step 2](../2_BaselineModel/baseline_model.ipynb)) classifies each **frame** as GOOD or BAD from labelled examples of both. The LSTM Autoencoder ([notebook](LSTM%20Autoencoder.ipynb)) trains on good **strokes** only and scores anything it cannot reconstruct as anomalous. The headline metrics are therefore not directly comparable; what is comparable is how each behaves under a validation split that holds out a whole recording.

| Model | Optimistic split | Held-out recording |
|---|---|---|
| DummyClassifier | 0.836 accuracy | — |
| Random Forest | 0.981 accuracy, 0.998 ROC-AUC | 5.4 % of bad frames detected |
| LSTM Autoencoder | 0.987 ROC-AUC | mean ROC-AUC 0.620 |

### Improvement

On the optimistic split the two are indistinguishable — and both are misleading. The difference appears only under recording-level validation, where the Random Forest collapses to 5.4 % while the autoencoder stays above chance.

The reason is structural, not a matter of capacity. The Random Forest sees BAD labels, and since every bad stroke comes from one rower it learns to recognise **that recording** — a probe confirmed the features identify the rower with 99.6 % accuracy. The autoencoder never sees a bad stroke, so that shortcut cannot form. Framing the task as one-class classification is the actual improvement here.

### Setback

0.620 is still unusable, and it degrades exactly where a real system would operate: on a rower the model does not know. Four of nine held-out recordings score below chance, meaning an unfamiliar good recording is flagged more readily than an actual fault.

A diversity sweep explains why. As more rowers enter the training set, the error on an unseen good recording falls by 40 % — the model does generalize across body proportions. But the error on bad strokes falls just as fast, and the gap the detector lives on shrinks by two thirds. **A bad stroke is still rowing**: reconstructing rowing better means reconstructing faulty rowing better too. The anomaly score was largely reporting novelty, and novelty is what diversity removes.

### Conclusion

Neither model can currently be validated as a technique detector, and the cause is the same for both: with bad strokes from a single rower, "faulty technique" and "this person" are not separable. No architecture resolves that. The prerequisite is bad recordings from several rowers, all demonstrating the same fault catalogue — see the next steps in the autoencoder notebook.

Until then, the honest figures are the held-out-recording column, and the DummyClassifier at 0.836 accuracy remains the reference point.

In [ ]:
# Comparative Analysis code (if applicable)
# Example: comparing accuracy of the baseline model and the new model
# print(f"Baseline Model Accuracy: {baseline_accuracy}, New Model Accuracy: {new_model_accuracy}")
